# Kauvar, Richman, Liu et al. (2025): human iEEG spectral response (Figs. 2G to 2J, 3B, 3C; figs. S12, S13, S15, S27)

Paper: Kauvar IV, Richman EB, Liu TX et al. (2025), Conserved brain-wide emergence of emotional response
from sensory experience in humans and mice. Science 388, eadt3971,
[doi:10.1126/science.adt3971](https://doi.org/10.1126/science.adt3971). The claims file
(claims/001347/10.1126-science.adt3971.yaml) was written from the version of record with its
supplementary materials.

Dataset: [DANDI:001347](https://dandiarchive.org/dandiset/001347/0.250528.0702), version
0.250528.0702, streamed. The paper cites the dandiset without a version; this is the only published
version (28 May 2025), so it is pinned.

What this notebook uses: the aggregated human products in `sub-H/` (per-channel trial means split into
train and test halves, per-channel, per-network and per-region per-trial spectrograms), and, to compare,
the paper's own deposited outputs in the same folder (NMF factors and loadings, cluster-test maps). The
Yeo7 network of each channel is not in the dataset; it is read from the paper's code record (Zenodo
14920570, `anat/pts_ch_labels.pkl`) and checked against the dataset in the link claim
`link-ieeg-yeo-labels`. Nothing random is drawn: NMF uses random_state 1 and the cluster tests seed 1,
as in the paper's code; the label permutation of figs27 is enumerated exactly. figs15 and figs27 use the
per-participant 20-min LFP files and, for figs27, the CADSS scores of the code record (`data/cadss_across_pts.csv`,
link claim `cadss-scores-link` in notebook 01).

| Claim | Data | Status |
|---|---|---|
| fig2g-nmf-six-factors | dataset | tested |
| link-ieeg-yeo-labels | with labels | tested (link claim) |
| fig2h-factor-network-loading | with labels | tested |
| fig2i-ketamine-reduces-factors-1-2 | dataset | tested |
| fig2j-factor-change-by-network | with labels | tested |
| figs13-auditory-control-no-response | dataset | tested |
| fig3b-cluster-test-yeo-networks | dataset | tested |
| fig3c-cluster-test-regions | dataset | tested |
| figs15-ketamine-spectral-changes-frontal | dataset | tested (approximate windows) |
| figs12-erp-not-explained-by-eyelid-closure | dataset | blocked (data_missing) |
| figs27-slow-oscillations-with-strong-dissociation | dataset, CADSS from the code record | tested (approximate windows) |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

import dandi_io
import ieeg_spectral as ie

warnings.filterwarnings("ignore")  # sklearn ConvergenceWarning and statsmodels fit notes; iterations are reported
pd.set_option("display.width", 200)
np.set_printoptions(precision=3, suppress=True, linewidth=160)
T0 = time.time()
claims = {c["id"]: c for c in yaml.safe_load(open("../../claims/001347/10.1126-science.adt3971.yaml"))["claims"]}
print("dandiset", dandi_io.DANDISET, "version", dandi_io.VERSION)

## Data check

Checks made below, each against what the claims file and the paper say:
- The per-channel file holds 458 channels from 7 participants with the counts of Table S3 (SD168 86,
  SD172 97, SD181 58, SD185 43, SD186 47, SD211 44, SD245 83), all in the 21 regions of Fig. 3C.
- Its train and test arrays are exactly the means of the odd-numbered and even-numbered trials
  (code split 1 and split 0) of the per-trial file, over the bins 0 < t < 1.4 s, for one channel per
  participant and each session. This ties the two files together and fixes what "train" and "test" are.
- Trials per participant and session (the per-trial file pads with NaN to 39), which participants have an
  auditory-control session, and that the per-network and per-region files pair infusion and preinfusion
  trials by participant and trial index (same labels row by row).
- Frequency axis 0 to 100 Hz in 65 bins; time axes; values are z-scores (no unit is stored).
- A clock check: the per-network preinfusion spectrograms show the puff response after t = 0 and not before.
- What the per-participant LFP and behaviour files hold, for figs15, figs27 and figs12.

In [ ]:
C = ie.load_channel_means()
chs = C["channels"]; n_ch = len(chs)
pts = np.array([c.split("_")[1] for c in chs]); regs = np.array([c.split("_")[0] for c in chs])
print("per-channel file:", {k: C[k].shape for k in ie.SPLITS})
print("value range (z):", {k: (round(float(C[k].min()), 2), round(float(C[k].max()), 2)) for k in ie.SPLITS})
print("NaN values:", sum(int(np.isnan(C[k]).sum()) for k in ie.SPLITS))
print("frequencies (Hz):", C["ff"][0], "to", C["ff"][-1], len(C["ff"]), "bins, step", np.diff(C["ff"])[0])
print("time (s, from the first bin after the puff):", C["tt"][0], "to", round(C["tt"][-1], 3), len(C["tt"]), "bins")
print("channels:", n_ch, "participants:", dict(sorted(Counter(pts).items())))
print("regions:", len(set(regs)), "all in the 21 regions of Fig. 3C:", set(regs) <= set(ie.PLOT_REGIONS))
print("channels per region:", dict(Counter(regs).most_common()))

s_tr, names_tr, tt_tr = ie.open_trial_file()
pos_tr = {c: i for i, c in enumerate(names_tr)}
print("\nper-trial file: channels", len(names_tr), "by participant", dict(sorted(Counter(c.split("_")[1] for c in names_tr).items())))
print("  shape per session", s_tr["puff_sgram_preinfusion"].shape, "time", tt_tr[0], "to", tt_tr[-1], "s,", len(tt_tr), "bins")
print("  the 458 channels are all in it:", set(chs) <= set(names_tr),
      "; labels outside the 21 regions (white matter, mixed labels) among the other", len(names_tr) - n_ch)
TIND = np.where((tt_tr > 0) & (tt_tr < 1.4))[0]
print("  NMF bins 0 < t < 1.4 s:", TIND[0], "to", TIND[-1], len(TIND), "bins; first bin", round(tt_tr[TIND[0]], 4),
      "s, so the per-channel file's time axis is t - ", round(tt_tr[TIND[0]], 4))
rows = []
for pt in sorted(set(pts)):
    i = np.where(pts == pt)[0][0]
    row = dict(participant=pt, channel=chs[i])
    for sess, key in [("preinfusion", "pre"), ("infusion", "inf"), ("postinfusion", "post")]:
        x = ie.read_channel_trials(s_tr, pos_tr[chs[i]], sess)
        row[f"n_{key}"] = x.shape[0]
        d_tr = np.abs(x[1::2][:, :, TIND].mean(0) - C[f"{key}_train"][i]).max()
        d_te = np.abs(x[0::2][:, :, TIND].mean(0) - C[f"{key}_test"][i]).max()
        row[f"maxdiff_{key}"] = max(d_tr, d_te)
    row["n_control"] = ie.read_channel_trials(s_tr, pos_tr[chs[i]], "control").shape[0]
    rows.append(row)
trials = pd.DataFrame(rows).set_index("participant")
print(trials.to_string())
print("total trials per session:", trials[["n_pre", "n_inf", "n_post", "n_control"]].sum().to_dict())
CTRL_PTS = trials.index[trials.n_control > 0].tolist()
print("participants with an auditory control:", CTRL_PTS)

In [ ]:
YEO = ie.load_group_file(ie.F_YEO)
REG = ie.load_group_file(ie.F_REG)
for lab, G in [("network", YEO), ("region", REG)]:
    print(f"per-{lab} file: {len(G['units'])} units; time {G['tt'][0]:.3f} to {G['tt'][-1]:.3f} s ({len(G['tt'])} bins); "
          f"freq {G['ff'][0]} to {G['ff'][-1]} Hz ({len(G['ff'])} bins)")
    paired = all((G["sess_pts"][(u, "preinfusion")] == G["sess_pts"][(u, "infusion")]).all() and
                 (G["sess_tri"][(u, "preinfusion")] == G["sess_tri"][(u, "infusion")]).all() for u in G["units"])
    print("  infusion and preinfusion rows carry the same participant and trial index in every unit:", paired)
print("network file time bins against per-trial bins 29 to 68, max difference (s):", round(float(np.abs(YEO["tt"] - tt_tr[29:69]).max()), 5))
ntr = pd.DataFrame({u: {"trials": len(YEO["sess_pts"][(u, "preinfusion")]),
                        "participants": len(set(YEO["sess_pts"][(u, "preinfusion")]))} for u in YEO["units"]}).T
print("trials per network (preinfusion):\n", ntr.to_string())
ntr_r = pd.DataFrame({u: {"trials": len(REG["sess_pts"][(u, "preinfusion")]),
                          "participants": len(set(REG["sess_pts"][(u, "preinfusion")]))} for u in REG["units"]}).T
print("trials per region (preinfusion), the 21 of Fig. 3C:\n", ntr_r.loc[ie.PLOT_REGIONS].T.to_string())
print("other regions in the file:", [u for u in REG["units"] if u not in ie.PLOT_REGIONS])

# clock check: mean |z| over trials and frequencies before and after the puff, preinfusion networks
pre_y = YEO["h"]["scratch/puff_region_sgram_preinfusion"][:]
m = np.nanmean(pre_y, axis=1)  # networks x freq x time
before = np.abs(m[:, :, YEO["tt"] < 0]).mean((1, 2)); after = np.abs(m[:, :, (YEO["tt"] > 0) & (YEO["tt"] < 1.4)]).mean((1, 2))
print("clock check, mean |z| of the trial mean, before / after the puff:")
for u, b, a in zip(YEO["units"], before, after):
    print(f"  {u:18s} {b:.3f} / {a:.3f}")

In [ ]:
# Per-participant LFP and behaviour files: what they hold (for figs15, figs27, figs12)
assets = dandi_io.list_assets()
ec = sorted(p for p in assets if p.startswith("sub-SD") and p.endswith("_ecephys.nwb"))
rows = []
for p in ec:
    h = dandi_io.open_h5(p)
    v = h["processing/ecephys/LFP/v_trace"]
    rate = v["starting_time"].attrs["rate"]
    rows.append(dict(file=p.split("/")[1], minutes=v["data"].shape[0] / rate / 60, rate=rate,
                     channels=v["data"].shape[1], description=h["session_description"][()].decode()[:60],
                     intervals="intervals" in h, epochs="intervals/epochs" in h,
                     top_groups=",".join(k for k in h.keys() if k not in ("specifications", "file_create_date", "general",
                                                                          "identifier", "session_description",
                                                                          "session_start_time", "timestamps_reference_time"))))
EC = pd.DataFrame(rows)
print(EC.to_string())
print("durations (min):", sorted(set(EC.minutes.round(2))))
b = dandi_io.open_h5("sub-SD245/sub-SD245_ses-puff_behavior.nwb")
print("\nbehaviour file sub-SD245_ses-puff_behavior.nwb, acquisition:", list(b["acquisition"].keys()))
print("top-level groups:", [k for k in b.keys() if k != "specifications"], "processing:", list(b["processing"].keys()))
print("any name with 'blink' or 'cadss' in any behaviour file:")
hits = []
for p in sorted(x for x in assets if x.startswith("sub-SD") and x.endswith("_behavior.nwb")):
    h = dandi_io.open_h5(p)
    h.visit(lambda name: hits.append((p, name)) if ("blink" in name.lower() or "cadss" in name.lower()) else None)
print(hits if hits else "none")

The per-participant LFP files are 20-min recordings (10 min before and after the first puff of each
session) at 200 Hz; they hold no infusion start time, no epochs or annotations of sleep or seizure
activity, and the behaviour files hold puff-aligned eye-closure traces, puff events and trial lists but
no spontaneous blink onsets and no CADSS scores (printed above). The infusion file therefore covers
about minutes 20 to 40 of the infusion (sections figs15 and figs27); the missing blink onsets block
figs12 (end).

## fig2g-nmf-six-factors

> Non-negative matrix factorisation of trial-averaged, puff-triggered channel spectrograms yields six shared
> spectrotemporal factors, with explained variance 0.47 on held-out trials, from 458 channels of 7 participants.

The training set is the mean of the preinfusion and postinfusion training halves (odd trials), the test
set the same for the even trials, as in the code (`compute_factors_using = 'prepost'`). NMF is fitted with
k = 1 to 14 components per part (2 to 28 in all), as in fig. S17A. Held-out explained variance is read two
ways: (a) the code's, the training reconstruction compared with the test data; (b) loadings refitted
on the test data. The refit with k = 3 is compared with the paper's deposited factors and loadings.

In [ ]:
X_tr = ((C["pre_train"] + C["post_train"]) / 2).reshape(n_ch, -1)
X_te = ((C["pre_test"] + C["post_test"]) / 2).reshape(n_ch, -1)
SHAPE = C["pre_train"].shape[1:]
t = time.time()
ev = ie.explained_variance_curve(X_tr, X_te, range(1, 15))
print(f"fits took {time.time() - t:.1f} s")
print(ev.round(3).to_string(index=False))
print("fits stopping at the 200-iteration default (not converged), positive / negative part:",
      int((ev.iter_pos >= 200).sum()), "/", int((ev.iter_neg >= 200).sum()), "of", len(ev))

In [ ]:
exp = claims["fig2g-nmf-six-factors"]["expected"][0]
six = ev.set_index("n_components").loc[6]
print(f"test explained variance, six components: code reading {six.r2_test_code:.3f}, refit reading {six.r2_test_refit:.3f}; "
      f"expected {exp['value']} +/- {exp['tolerance']}")
gains = ev.set_index("n_components")[["r2_test_code", "r2_test_refit", "r2_train"]].diff()
print("gain from each step of two components:\n", gains.round(3).T.to_string())
for col in ["r2_test_code", "r2_test_refit", "r2_train"]:
    y = ev[col].values; x = ev.n_components.values
    g = np.diff(y)                      # gain into x[1:]
    drop = g[:-1] - g[1:]               # fall in gain after x[1:-1]
    print(f"{col}: elbow by largest fall in gain = {x[1:-1][np.argmax(drop)]}, "
          f"by largest distance from the chord = {ie.elbow(x, y):.0f}")
# the same rules over a shorter range of k, to show their dependence on it
for kmax in [8, 10, 12]:
    sub = ev[ev.k <= kmax]
    print(f"  k up to {kmax}: chord elbow of r2_test_code = {ie.elbow(sub.n_components, sub.r2_test_code):.0f}")

mp, mn = ie.fit_nmf_pair(X_tr, 3)
cc = ie.components(mp, mn, SHAPE)
FAC, LOAD = ie.load_deposited_nmf()
print("\nrefit (k = 3 per part) against the deposited factors, in the code's order", ie.CODE_ORDER)
for i, fi in enumerate(ie.CODE_ORDER):
    print(f"  factor {i} = fitted {fi}: max |difference| {np.abs(cc[fi] - FAC[i]).max():.1e}, r = {np.corrcoef(cc[fi].ravel(), FAC[i].ravel())[0, 1]:.6f}")
print("deposited channel list equals the per-channel file's:", (LOAD["channels"] == chs).all())

# per-participant breakdown of the code-reading test explained variance at six components
H = np.vstack([mp.components_, -mn.components_])
rec_tr = ie.transform(mp, mn, X_tr) @ H
per_pt = {pt: round(ie.r2(X_te[pts == pt], rec_tr[pts == pt]), 3) for pt in sorted(set(pts))}
print("test explained variance (code reading) by participant:", per_pt)

fig, axs = plt.subplots(1, 7, figsize=(16, 2.4))
axs[0].plot(ev.n_components, ev.r2_test_code, ".-", label="test (code)")
axs[0].plot(ev.n_components, ev.r2_test_refit, ".-", label="test (refit)")
axs[0].plot(ev.n_components, ev.r2_train, ".-", label="train"); axs[0].legend(fontsize=6); axs[0].set_xlabel("components")
for i, fi in enumerate(ie.CODE_ORDER):
    axs[i + 1].imshow(cc[fi], origin="lower", aspect="auto", cmap="PRGn", vmin=-1, vmax=1,
                      extent=[C["tt"][0], C["tt"][-1], C["ff"][0], C["ff"][-1]])
    axs[i + 1].set_title(f"factor {i}", fontsize=8)
plt.tight_layout(); plt.show()
print("factor value range:", round(float(cc.min()), 2), round(float(cc.max()), 2), "(colour scale -1 to 1)")

Open G, reading (b): factor 1 is the positive factor with the most energy above 50 Hz, factor 2 the
negative factor with the most energy below 20 Hz, the others numbered within their part by peak time.

In [ ]:
ORDER_B, desc = ie.description_order(cc, C["ff"], C["tt"])
print(pd.DataFrame(desc, index=[f"fitted {i}" for i in range(6)]).round(3).to_string())
print("reading (b) order:", ORDER_B, " code order:", ie.CODE_ORDER, " same:", ORDER_B == ie.CODE_ORDER)

**Result: reproduced, resting on the choice of elbow rule.** Refitting NMF (three positive and three
negative components) on the pre/postinfusion training means of the 458 channels from 7 participants gives
the deposited factors to within 3e-13 and the deposited loadings exactly (next sections). The held-out
explained variance with six components is 0.466 under the code's reading (training reconstruction
against the test data) and 0.535 with loadings refitted on the test data; the first is within
0.47 +/- 0.047. The test curve gains 0.081 and 0.056 in the steps to 4 and 6 components and 0.014 or
less in every later step. The largest fall in gain is at 6 components under both readings; the point
farthest from the chord is at 8 over the full range of 2 to 28 components and at 6 when the range ends
at 16, 20 or 24 components. The paper names no rule; the largest fall in gain, the usual meaning of an
elbow, is assumed. By participant the test explained variance runs from 0.292 (SD186) to 0.654 (SD211).
Both Open G readings number the factors the same way. Of the 28 fits, 12 stop at the default 200
iterations, as the paper's code would.

## link-ieeg-yeo-labels (link claim)

The Yeo7 network of each bipolar channel is not in the dataset. The code reads it from
`anat/pts_ch_labels.pkl` (key `pt_elec_yeos_bipo`), matching channels by participant and bipolar electrode
name. The link holds when every one of the 458 channels is found, its atlas region in the pickle equals
the region in the dataset's channel name, the dataset's own per-network electrode counts
(`anatomy_data` of the per-network file, by participant, network and region) agree with the pickle,
and the labels turn the deposited channel loadings into the deposited network loadings of Fig. 2H.

In [ ]:
labels, md5, member = ie.fetch_code_labels()
print("fetched", member, "md5", md5)
yeo, anat = ie.yeo_labels(chs, labels)
print("channels found:", int((yeo != "Missing").sum()), "of", n_ch)
print("atlas region equals the channel-name region:", int(sum(a == r for a, r in zip(anat, regs))), "of", n_ch)
in_net = np.isin(yeo, ie.YEO_REGIONS)
print("channels in one of the six networks:", int(in_net.sum()), dict(Counter(yeo[in_net]).most_common()))
print("other labels (excluded from network means, as in the code):", int((~in_net).sum()), dict(Counter(yeo[~in_net]).most_common(8)), "...")

# counts by participant, network and region: dataset per-network file against the pickle
rows = []
for pt in sorted(set(pts)):
    y = np.array(labels["pt_elec_yeos_bipo"][pt]); a = np.array(labels["pt_elec_anats_bipo"][pt])
    for net in ie.YEO_REGIONS:
        for k, v in Counter(a[y == net]).items():
            rows.append(dict(patient=pt, region=net, anat=k, n_pkl=v))
cmp = YEO["anat"].merge(pd.DataFrame(rows), on=["patient", "region", "anat"], how="outer")
same = (cmp.nelec == cmp.n_pkl)
print(f"\nper-network file rows: {len(YEO['anat'])}; equal counts in the pickle: {int(same.sum())}; "
      f"rows where the pickle has more electrodes: {int((cmp.n_pkl > cmp.nelec).sum() + cmp.nelec.isna().sum())}; "
      f"rows where the dataset has more: {int((cmp.nelec > cmp.n_pkl).sum() + cmp.n_pkl.isna().sum())}")
print(cmp[~same].to_string(index=False))

# deposited network loadings recomputed from the deposited channel loadings with these labels
dep_pre = ie.group_means(LOAD["pre"], yeo, ie.YEO_REGIONS, normalize_row=True)
dep_d = ie.group_means(LOAD["inf"] - LOAD["pre"], yeo, ie.YEO_REGIONS)
print("deposited network order:", LOAD["yeo"])
print("max |difference|, Fig. 2H matrix:", float(np.nanmax(np.abs(dep_pre.values - LOAD["yeo_pre"]))),
      "; Fig. 2J matrix:", float(np.nanmax(np.abs(dep_d.values - LOAD["yeo_inf_minus_pre"]))))

**Result: the link holds.** All 458 channels are found in the code record's label file, and their atlas
region equals the region in the dataset's channel name for 458 of 458. Of the 126 participant, network and
region counts in the dataset's per-network file, 120 equal the label file's; in the other 14 rows the
label file has more electrodes (including labels marked EXCLUDE), never fewer, as expected if channels
removed before the dataset's aggregates were built are still listed in the label file. With these labels
the deposited channel loadings reproduce the deposited Fig. 2H and Fig. 2J matrices to 6e-17. Yeo7
labels are missing from the dataset for every channel; only 221 of the 458 channels carry a single
network label and enter the network means.

## fig2h-factor-network-loading

> Factors 0 to 2 load mostly on attentional and somatomotor networks, and factors 3 to 5 mostly on limbic,
> default-mode and frontoparietal networks.

Preinfusion test loadings, thresholded as in the code, averaged over each network's channels, each row
divided by its sum. Both Open G readings give the same numbering (above), so one table serves both.

In [ ]:
W_raw = {"pre": ie.transform(mp, mn, C["pre_test"].reshape(n_ch, -1)),
         "inf": ie.transform(mp, mn, C["inf_test"].reshape(n_ch, -1)),
         "post": ie.transform(mp, mn, C["post_test"].reshape(n_ch, -1))}
W_pp = ie.transform(mp, mn, ((C["pre_test"] + C["post_test"]) / 2).reshape(n_ch, -1))
Wt, EXCL = ie.threshold_loadings(W_raw, W_pp)
W = {k: v[:, ie.CODE_ORDER] for k, v in Wt.items()}          # columns are factors 0..5
EXCL = EXCL[:, ie.CODE_ORDER]
print("refit loadings equal the deposited ones (NaN in the same places):",
      all(np.allclose(W[k], LOAD[k], equal_nan=True, atol=1e-10) for k in ["pre", "inf", "post"]))
print("channels kept per factor (loading of the pre/post test mean >= 0.1):", (~EXCL).sum(0))

M = ie.group_means(W["pre"], yeo, ie.YEO_REGIONS, normalize_row=True)
M.columns = [f"f{i}" for i in range(6)]
M["share f0-f2"] = M[["f0", "f1", "f2"]].sum(1); M["share f3-f5"] = M[["f3", "f4", "f5"]].sum(1)
M["channels"] = [int((yeo == r).sum()) for r in ie.YEO_REGIONS]
print(M.round(3).to_string())
want = {"Dorsal Attention": "share f0-f2", "Ventral Attention": "share f0-f2", "Somatomotor": "share f0-f2",
        "Limbic": "share f3-f5", "Default": "share f3-f5", "Frontoparietal": "share f3-f5"}
ok = {r: bool(M.loc[r, c] > 0.5) for r, c in want.items()}
print("rows meeting the criterion:", ok, "->", sum(ok.values()), "of 6")

# participant breakdown: share of factors 0-2 per network, per participant
rows = {}
for pt in sorted(set(pts)):
    sel = pts == pt
    Mp = ie.group_means(W["pre"][sel], yeo[sel], ie.YEO_REGIONS, normalize_row=True)
    rows[pt] = Mp.iloc[:, :3].sum(1).where(Mp.notna().any(axis=1))
print("share of factors 0-2 by participant (NaN: no channel):\n", pd.DataFrame(rows).round(2).to_string())

**Result: partially reproduced.** Factors 0 to 2 carry more than half of the dorsal attention (0.671),
ventral attention (0.651) and somatomotor (0.662) rows, and factors 3 to 5 more than half of the default
row (0.536). They do not in the limbic row (factors 3 to 5 carry 0.495) or the frontoparietal row (0.463).
Four of the six rows meet the criterion, the partial range. The loadings are identical to the deposited
ones and the matrix to the deposited Fig. 2H matrix, so the two failing rows are in the paper's own
numbers. Dorsal attention rests on 8 channels; by participant the share of factors 0 to 2 in the limbic
row runs from 0.27 to 0.73.

## fig2i-ketamine-reduces-factors-1-2

> Factors 1 and 2 are reduced during ketamine and recover after infusion; in the comparison of infusion with
> the mean of preinfusion and postinfusion, factors 0, 4 and 5 show no significant change.

Mixed model of loading on condition with channels grouped by participant (statsmodels mixedlm), infusion
against the mean of pre and post, Benjamini-Hochberg across the six factors; post-hoc preinfusion against
infusion and postinfusion against infusion, corrected together (12 tests, as the code's
`comparison_list = ['pre', 'post']` does). Coefficients are infusion minus the comparison.

In [ ]:
from statsmodels.stats.multitest import multipletests
main, post_hoc = [], []
for f in range(6):
    r = ie.mixedlm_compare((W["pre"][:, f] + W["post"][:, f]) / 2, W["inf"][:, f], pts); r["factor"] = f; main.append(r)
    for lab, ref in [("pre", W["pre"][:, f]), ("post", W["post"][:, f])]:
        r = ie.mixedlm_compare(ref, W["inf"][:, f], pts); r.update(factor=f, against=lab); post_hoc.append(r)
main = pd.DataFrame(main).set_index("factor"); main["p_bh"] = multipletests(main.p, method="fdr_bh")[1]
post_hoc = pd.DataFrame(post_hoc); post_hoc["p_bh"] = multipletests(post_hoc.p, method="fdr_bh")[1]
means = pd.DataFrame({k: np.nanmean(W[k], 0) for k in ["pre", "inf", "post"]})
main = main.join(means)
fmt = lambda x: f"{x:.3g}"
print("infusion against mean of pre and post:\n", main.to_string(float_format=fmt))
print("\npost hoc (coefficient = infusion - comparison):\n", post_hoc.set_index(["factor", "against"]).to_string(float_format=fmt))
print("\nkept channels by participant and factor:\n", pd.DataFrame({f"f{f}": pd.Series(~np.isnan(W["pre"][:, f])).groupby(pts).sum() for f in range(6)}).to_string())
exp = claims["fig2i-ketamine-reduces-factors-1-2"]["expected"]
meas = [main.loc[1, "pre"], main.loc[1, "inf"], main.loc[1, "post"], main.loc[2, "pre"], main.loc[2, "inf"], main.loc[2, "post"]]
for e, m_ in zip(exp, meas):
    print(f"  {e['quantity']}: {m_:.3f} (expected {e['value']} +/- {e['tolerance']}) {'within' if abs(m_ - e['value']) <= e['tolerance'] + 1e-9 else 'outside'}")
print("\nmean loading by participant and session, factors 1 and 2:")
for f in [1, 2]:
    tab = pd.DataFrame({k: pd.Series(W[k][:, f]).groupby(pts).mean() for k in ["pre", "inf", "post"]})
    tab["n_ch"] = pd.Series(~np.isnan(W["pre"][:, f])).groupby(pts).sum()
    print(f"factor {f}:\n", tab.round(3).to_string())
print("\nparticipants with kept channels per factor, here against the claims file's statistics (Table S3):")
st = {s["about"].split(",")[0]: s["n"] for s in claims["fig2i-ketamine-reduces-factors-1-2"]["statistics"] if "n" in s}
for f in range(6):
    kept = pd.Series(~np.isnan(W["pre"][:, f])).groupby(pts).sum()
    print(f"  factor {f}: {int((kept > 0).sum())} participants, {int(kept.sum())} channels, fewest per participant "
          f"{int(kept.min())} ({kept.idxmin()}); Table S3: {st[f'factor {f}']}")


**Result: reproduced.** Against the mean of preinfusion and postinfusion, infusion lowers factor 1
(coefficient -0.125, corrected p 2.56e-05, 260 channels) and factor 2 (-0.115, 5.51e-08, 185 channels).
Factors 0 (-0.049, 0.121), 4 (0.013, 0.406) and 5 (0.013, 0.341) are not significant; factor 3 is
(-0.041, 0.0159, model not converged). Post hoc, infusion is below postinfusion for factor 1 (-0.061,
corrected p 0.0464) and factor 2 (-0.089, 5.16e-05), the recovery the claim states. The six means are
within tolerance (factor 1: 0.508, 0.318, 0.379; factor 2: 0.354, 0.214, 0.302). Every coefficient,
corrected p and channel count equals Table S3 to its precision; the signs of the post-hoc coefficients
are reversed there (Table S3 reports the comparison minus infusion). Table S3 gives 5 participants for
factor 3 and 6 for factor 4 with the same channel counts (158 and 370); here all 7 participants keep
channels for every factor, the fewest being 13 (SD186) for factor 3 and 26 (SD211) for factor 4, so the
difference is not a participant with few channels. Factor 1
falls from preinfusion to infusion in all 7 participants and factor 2 in 6 of 7 (SD181 rises from 0.199
to 0.294). As Table S3 also shows, the post-hoc tests are significant for factor 0 (pre, 0.036), factor 4
(pre, 0.000299; post, 0.0464) and factor 5 (pre, 0.00985).

## fig2j-factor-change-by-network

> During ketamine, factor 1 is suppressed in limbic, dorsal attention and ventral attention networks, and
> factor 2 in somatomotor, dorsal attention and ventral attention networks.

Change in loading (infusion minus preinfusion) averaged over each network's channels, channels kept as in
fig2i. Readings of "suppressed": (a) negative in each named network, (b) the named networks are the three
most negative.

In [ ]:
D = ie.group_means(W["inf"] - W["pre"], yeo, ie.YEO_REGIONS)
D.columns = [f"f{i}" for i in range(6)]
nch = pd.DataFrame({f"n_f{f}": pd.Series(~np.isnan(W["pre"][:, f])).groupby(yeo).sum().reindex(ie.YEO_REGIONS) for f in [1, 2]})
print(D.round(3).join(nch).to_string())
named = {1: ["Limbic", "Dorsal Attention", "Ventral Attention"], 2: ["Somatomotor", "Dorsal Attention", "Ventral Attention"]}
for f, nets in named.items():
    col = D[f"f{f}"]
    a = all(col[n] < 0 for n in nets)
    b = set(col.nsmallest(3).index) == set(nets)
    print(f"factor {f}: (a) negative in all named: {a}; (b) named are the three most negative: {b} (three most negative: {list(col.nsmallest(3).index)})")
print("\nchange by participant, factor 1 and 2, named networks (NaN: no kept channel):")
for f, nets in named.items():
    rows = {}
    for pt in sorted(set(pts)):
        sel = pts == pt
        rows[pt] = ie.group_means((W["inf"] - W["pre"])[sel], yeo[sel], nets).iloc[:, f]
    print(f"factor {f}:\n", pd.DataFrame(rows).round(3).to_string())

**Result: reproduced under reading (a).** Infusion minus preinfusion, factor 1 is negative in the limbic
(-0.184), dorsal attention (-0.225) and ventral attention (-0.266) networks, and factor 2 in the
somatomotor (-0.108), dorsal attention (-0.190) and ventral attention (-0.157) networks. Under reading (b)
factor 2 holds and factor 1 does not: its three most negative networks are default (-0.269), ventral
attention and dorsal attention, with limbic fourth; factor 1 falls in all six networks (-0.083 to -0.269).
The dorsal attention values rest on 7 channels from 3 participants (factor 1) and 5 channels from 2
participants (factor 2). The matrix equals the deposited Fig. 2J matrix (link section).

## figs13-auditory-control-no-response

> The brain-wide eyepuff-evoked response is absent in the auditory control, where the device sounds but the
> air is directed away from the participant.

The control session's trial mean (all trials, as the code's `X_ctrl`) of each channel of the participants
with a control session is read from the per-trial file and projected on the fig2g factors; channels
excluded for a factor in fig2i are excluded here too (code `W_ctrl`). The criterion compares the mean
loading over channels in the control with the preinfusion mean; both the preinfusion mean over the same
channels and over all channels are shown. The preinfusion loadings are those of the test half of the trials
(fig2i); as a variant, preinfusion loadings from all preinfusion trials, the same trial use as the control.

In [ ]:
t = time.time()
ctrl_idx = np.where(np.isin(pts, CTRL_PTS))[0]
Xc = np.stack([ie.read_channel_trials(s_tr, pos_tr[chs[i]], "control")[:, :, TIND].mean(0) for i in ctrl_idx])
print(f"read control trial means of {len(ctrl_idx)} channels in {time.time() - t:.0f} s; NaN: {int(np.isnan(Xc).sum())}")
Wc = ie.transform(mp, mn, Xc.reshape(len(ctrl_idx), -1))[:, ie.CODE_ORDER]
Wc[EXCL[ctrl_idx]] = np.nan
res = pd.DataFrame({"control": np.nanmean(Wc, 0), "pre (same channels)": np.nanmean(W["pre"][ctrl_idx], 0),
                    "pre (all channels)": np.nanmean(W["pre"], 0), "n_ch": (~np.isnan(Wc)).sum(0)})
res["control lower than pre (same channels)"] = res["control"] < res["pre (same channels)"]
res["ratio control / pre"] = res["control"] / res["pre (same channels)"]
print(res.round(3).to_string())
print("factors with lower control loading:", int(res["control lower than pre (same channels)"].sum()), "of 6")
print("\nmean loading by participant, control / preinfusion (same channels):")
cp = pts[ctrl_idx]
tab = pd.DataFrame({f"f{f}": [f"{np.nanmean(Wc[cp == p, f]):.2f}/{np.nanmean(W['pre'][ctrl_idx][cp == p, f]):.2f}" for p in CTRL_PTS] for f in range(6)}, index=CTRL_PTS)
print(tab.to_string())
lower_pt = np.array([[np.nanmean(Wc[cp == p, f]) < np.nanmean(W["pre"][ctrl_idx][cp == p, f]) for f in range(6)] for p in CTRL_PTS])
print("participant-factor pairs with lower control loading:", int(lower_pt.sum()), "of", lower_pt.size)
# variant: preinfusion loadings from all preinfusion trials, as the control uses all of its trials
t = time.time()
Xp_all = np.stack([ie.read_channel_trials(s_tr, pos_tr[chs[i]], "preinfusion")[:, :, TIND].mean(0) for i in ctrl_idx])
Wp_all = ie.transform(mp, mn, Xp_all.reshape(len(ctrl_idx), -1))[:, ie.CODE_ORDER]
Wp_all[EXCL[ctrl_idx]] = np.nan
print(f"\nvariant, preinfusion from all preinfusion trials (read in {time.time() - t:.0f} s):")
res2 = pd.DataFrame({"control": np.nanmean(Wc, 0), "pre (all trials)": np.nanmean(Wp_all, 0),
                     "pre (test half)": np.nanmean(W["pre"][ctrl_idx], 0), "n_ch": (~np.isnan(Wc)).sum(0)})
res2["control lower than pre (all trials)"] = res2["control"] < res2["pre (all trials)"]
res2["ratio control / pre (all trials)"] = res2["control"] / res2["pre (all trials)"]
print(res2.round(3).to_string())
print("factors with lower control loading:", int(res2["control lower than pre (all trials)"].sum()), "of 6")
lower_pt2 = np.array([[np.nanmean(Wc[cp == p, f]) < np.nanmean(Wp_all[cp == p, f]) for f in range(6)] for p in CTRL_PTS])
print("participant-factor pairs with lower control loading:", int(lower_pt2.sum()), "of", lower_pt2.size)
print("pairs where it is not (participant, factor, control, pre all trials):",
      [(str(p), f, round(float(np.nanmean(Wc[cp == p, f])), 2), round(float(np.nanmean(Wp_all[cp == p, f])), 2))
       for j, p in enumerate(CTRL_PTS) for f in range(6) if not lower_pt2[j, f]])


**Result: reproduced.** Over the 275 channels of the 5 participants with an auditory-control session
(SD181, SD185, SD186, SD211, SD245), the mean control loading is below the preinfusion mean of the same
channels for all six factors (ratios 0.33 to 0.70), and for 30 of 30 participant and factor pairs.

## fig3b-cluster-test-yeo-networks

> Permutation cluster tests find ketamine-altered spectrotemporal clusters in the limbic, ventral attention
> and somatomotor Yeo7 networks.

Per network, the infusion minus preinfusion difference of trials paired by participant and trial index
(the per-network file's rows), mne `permutation_cluster_1samp_test`, threshold the two-tailed t at
p = 0.01, 512 permutations, seed 1, -0.5 to 1.5 s. The lowest cluster p of each network (1 when no cluster
is below 0.01, as the code does) is Benjamini-Hochberg corrected across the six networks. The deposited
change maps and contours are compared with these. Cluster-forming thresholds of p = 0.05 and 0.001 (the
code's PARAMS 1 and 2) are shown as variants. With preinfusion loadings from all preinfusion trials (the
control's trial use), the control mean is still lower for all six factors, by less (ratios 0.34 to 0.98;
factor 5 0.299 against 0.306), and for 24 of 30 participant and factor pairs (the six exceptions are factors
2 and 5).

In [ ]:
DEP = ie.load_deposited_clusters()
def run_units(G, units, thresh_p=0.01):
    out = {}
    for u in units:
        j = G["units"].index(u)
        pre = G["h"]["scratch/puff_region_sgram_preinfusion"][j]; inf = G["h"]["scratch/puff_region_sgram_infusion"][j]
        v = ~np.isnan(pre[:, 0, 0])
        r = ie.cluster_test(pre[v], inf[v], thresh_p=thresh_p)
        r["pts"] = G["sess_pts"][(u, "preinfusion")]; r["diff"] = inf[v] - pre[v]
        out[u] = r
    return out

t = time.time()
RY = run_units(YEO, ie.YEO_REGIONS)
qy = ie.fdr_units(RY, ie.YEO_REGIONS)
tab = pd.DataFrame({u: dict(trials=RY[u]["n_trials"], clusters=RY[u]["n_clusters"], signif_clusters=RY[u]["n_signif"],
                            min_p=RY[u]["min_p"], p_bh=qy[u], significant=qy[u] < 0.01,
                            dep_contour_px=int(DEP[u][1].sum()), contour_px=int(RY[u]["contour"].sum()),
                            T_vs_deposited_maxdiff=float(np.abs(DEP[u][0] - RY[u]["T"]).max()))
                    for u in ie.YEO_REGIONS}).T
print(f"({time.time() - t:.0f} s)\n", tab.to_string())
print("min p is bounded below by 1/512 =", round(1 / 512, 5))
for thr in [0.05, 0.001]:
    R2 = run_units(YEO, ie.YEO_REGIONS, thresh_p=thr); q2 = ie.fdr_units(R2, ie.YEO_REGIONS)
    print(f"variant threshold p = {thr}: significant networks", [u for u in ie.YEO_REGIONS if q2[u] < 0.01])
print("\nmean infusion - preinfusion change inside the significant clusters, by participant (z):")
rows = {}
for u in ie.YEO_REGIONS:
    if RY[u]["contour"].sum() == 0: continue
    m = RY[u]["diff"][:, RY[u]["contour"] > 0].mean(1)
    rows[u] = pd.Series(m).groupby(RY[u]["pts"]).mean()
print(pd.DataFrame(rows).round(3).to_string())

**Result: reproduced.** Limbic, ventral attention and somatomotor each have significant clusters
(lowest cluster p 0.00195, the floor for 512 permutations; corrected 0.0039), and frontoparietal (lowest
0.027), default (0.045) and dorsal attention (0.10) have none below 0.01. The t maps and contours equal the
deposited ones in all six networks (maximum difference 0). With a cluster-forming threshold of p = 0.05,
dorsal attention is significant as well; with 0.001, somatomotor is not. Inside the significant clusters
the limbic change is negative and the somatomotor change positive in each of the 7 participants; the
ventral attention mean over its two clusters is near zero in every participant (-0.101 to 0.072).

## fig3c-cluster-test-regions

> Ketamine alters the eyepuff spectrotemporal response in insula, cingulate, orbitofrontal, posteromedial,
> supramarginal and motor cortex and in hippocampus, basal ganglia and thalamus, while many other regions
> are unchanged.

The fig3b test on the 21 regions of Fig. 3C, corrected across these 21 (the criterion). As variants, the
correction across all 35 regions of the file and the deposited contours (33 regions) are shown. Reading (a)
of the criterion is run; reading (b) (adding the single-channel test) is needed only if (a) fails.

In [ ]:
t = time.time()
RR = run_units(REG, REG["units"])
q21 = ie.fdr_units(RR, ie.PLOT_REGIONS); q35 = ie.fdr_units(RR, REG["units"])
LISTED = ["INS", "ACC", "MCC", "ORB", "PMC", "SMG", "MOT", "HIPP", "BG", "THAL"]
tab = pd.DataFrame({u: dict(trials=RR[u]["n_trials"], participants=len(set(RR[u]["pts"])), min_p=RR[u]["min_p"],
                            p_bh_21=q21.get(u, np.nan), p_bh_35=q35[u],
                            dep_contour_px=int(DEP[u][1].sum()) if u in DEP else np.nan,
                            contour_px_21=int(RR[u]["contour"].sum()) if q21.get(u, 1) < 0.01 else 0,
                            T_vs_deposited_maxdiff=float(np.abs(DEP[u][0] - RR[u]["T"]).max()) if u in DEP else np.nan)
                    for u in REG["units"]}).T
print(f"({time.time() - t:.0f} s)")
print(tab.loc[ie.PLOT_REGIONS].to_string())
print("\nother regions (not in the 21):\n", tab.drop(ie.PLOT_REGIONS).to_string())
sig21 = {u for u in ie.PLOT_REGIONS if q21[u] < 0.01}
listed_ok = {l: (l in sig21) if l != "THAL" else ("THAL ANT" in sig21 or "THAL POS" in sig21) for l in LISTED}
others = [u for u in ie.PLOT_REGIONS if u not in ["INS", "ACC", "MCC", "ORB", "PMC", "SMG", "MOT", "HIPP", "BG", "THAL ANT", "THAL POS"]]
print("\nreading (a), listed regions significant:", listed_ok, "->", sum(listed_ok.values()), "of 10")
print("other regions of the 21:", others, "; not significant:", sum(u not in sig21 for u in others), "of", len(others))
print("  counting one thalamic region (ANT) among the listed and THAL POS among the others: not significant",
      sum(u not in sig21 for u in others + ["THAL POS"]), "of", len(others) + 1)
print("significant with correction across 35:", sorted(u for u in REG["units"] if q35[u] < 0.01))
print("regions with a deposited contour:", sorted(u for u in DEP if DEP[u][1].sum() > 0 and u not in ie.YEO_REGIONS))
for thr in [0.05, 0.001]:
    R2 = run_units(REG, ie.PLOT_REGIONS, thresh_p=thr); q2 = ie.fdr_units(R2, ie.PLOT_REGIONS)
    print(f"variant threshold p = {thr}: significant of the 21:", sorted(u for u in ie.PLOT_REGIONS if q2[u] < 0.01))
print("\nmean change inside the significant clusters by participant (z), listed regions:")
rows = {}
for u in sorted(sig21):
    m = RR[u]["diff"][:, RR[u]["contour"] > 0].mean(1)
    rows[u] = pd.Series(m).groupby(RR[u]["pts"]).mean()
print(pd.DataFrame(rows).round(3).to_string())

**Result: reproduced.** With correction across the 21 regions, all ten listed regions are significant
(INS, ACC, MCC, ORB, PMC, SMG, MOT, HIPP, BG, and both THAL ANT and THAL POS; lowest cluster p 0.00195,
corrected 0.0037), and none of the other ten (IFG, MFG, SS, PHG, STG, STS, MTG, ITG, TP, AMY) is; counting
THAL POS among the others, 10 of 11 are not significant, above the five required. The t maps equal the
deposited ones in all 33 deposited regions, and the significant set equals the regions with a deposited
contour, which add PRECG (one participant, outside the 21). ACC, MCC and MOT rest on 2 participants each
(61 to 70 trials). At a cluster-forming threshold of 0.001 all listed regions but THAL POS stay
significant; at 0.05, MTG and STG join. Reading (b) was not needed and not run.


## figs15-ketamine-spectral-changes-frontal

> Ketamine changes spectral power across the brain, with the largest changes in frontal cortex, led by
> orbitofrontal cortex.

An approximate test on the per-participant 20-min LFP files. Timing: the Methods (iEEG ketamine infusion
protocol) say "The eyepuff assay was administered preceding infusion, at approximately 30 minutes after the
infusion start, and 110 minutes after infusion start". Each file runs from 10 min before to 10 min after the
first puff, so the infusion file covers about minutes 20 to 40 of the infusion, and its second half (first
puff to the end) is the 30 to 40 min peak window of fig. S15C. Because the start is approximate and the
infusion start time is not in the dataset, the first half (about 20 to 30 min), the whole file, and the
second half without the eyepuff assay (from 2 s after the last puff) are run as well.

Baseline: the paper's 25-min pre-ketamine window does not fit in a 20-min file. The preinfusion eyepuff file
is used (the code's preinfusion session: Baseline for SD168, SD172 and SD245, Preinfusion for the others;
`ieeg_cd.PRE_SESSION_CODE`). Main: its first 10 min, before the first puff, since the Methods exclude the
eyepuff assay from preinfusion windows ("excluding periods of time in which the patient was undergoing the
eyepuff assay", Methods, phase locking); variants: the whole file, and SD245's Preinfusion file in place of
its Baseline file.

Spectrogram: multitaper with the parameters the Methods give for fig. S27 (time-half-bandwidth 30, 59 tapers,
30-s windows, 15-s stride); the paper gives none for fig. S15, so this is an assumption. Power is taken as
log10 (an assumption; linear power is a variant), z-scored per channel and frequency to the baseline
windows and averaged over the ketamine windows. Magnitude (criterion): per participant and region the mean
over channels of the mean |z| over 1 to 100 Hz (58 to 62 Hz, the notch, left out), then the median over
participants. Channels: within-shank bipolar pairs whose atlas label in the code record
(`pt_elec_anats_bipo`) is a single one of the 21 regions of Fig. 3C, without out-of-brain or excluded
contacts and without the code's bad channels (`utils.bad_ch`). The files carry no sleep or seizure
annotations, so no period is excluded for them. Cohorts: (a) the 7 included participants; (b) every
participant with a 40-min infusion in the dataset. Only the 7 included participants have LFP files, and the
dataset does not say which of them had a continuous 40-min infusion (the Methods mention target-controlled
and bolus infusions for some), so (b) is the same 7 or an unidentifiable subset of them; only (a) can be run.

In [ ]:
import ieeg_cd as cd
import human_eye as he
t = time.time()
BAD = cd.code_bad_channels(cd.fetch_code_zip())
inf_pts = sorted(p.split("/")[0][4:] for p in assets if p.endswith("_ses-puffInfusion_ecephys.nwb"))
print("participants with an infusion LFP file:", inf_pts, "; the 7 included:", inf_pts == sorted(cd.PARTICIPANTS))
REST = ie.rest_all(cd.PARTICIPANTS, labels, BAD)
print(f"streamed {sum(len(d) for d in REST.values())} files and computed their spectrograms in {time.time() - t:.0f} s")
rows = []
for pt, d in REST.items():
    for k, R in d.items():
        rows.append(dict(participant=pt, role=k, session=R["session"], minutes=R["duration"] / 60, first_puff_s=R["first_puff"],
                         last_puff_s=round(R["last_puff"], 1), puffs=R["n_puffs"], bipolar_pairs=R["n_pairs"], kept=len(R["ch"]),
                         windows=len(R["t"]), log10_power_range=f"{np.log10(R['S'].min()):.1f} to {np.log10(R['S'].max()):.1f}"))
print(pd.DataFrame(rows).to_string(index=False))
R0 = REST[cd.PARTICIPANTS[0]]["inf"]
print("frequency bins:", R0["f"][0], "to", R0["f"][-1], "Hz, step", ie.FBIN, "; window starts", R0["t"][0], "to", R0["t"][-1], "s")
for w in ["before", "after", "after_no_puffs", "whole"]:
    lo, hi = ie.window_bounds(R0, w)
    print(f"  {w:15s} {ie.REST_WINDOWS[w]}: {lo:.0f} to {hi:.0f} s in {R0['session']} of {cd.PARTICIPANTS[0]}, "
          f"{int(ie.window_mask(R0['t'], lo, hi).sum())} windows")
REG_TAB = pd.crosstab(pd.Series(np.concatenate([REST[p]["inf"]["region"] for p in cd.PARTICIPANTS]), name="region"),
                      pd.Series(np.concatenate([[p] * len(REST[p]["inf"]["ch"]) for p in cd.PARTICIPANTS]), name="participant"))
print("\nkept channels by region and participant:\n", REG_TAB.to_string())

In [ ]:
rows, F15 = [], {}
for base_key in ["pre", "pre_alt"]:
    for bw in ["before", "whole"]:
        for kw in ["after", "after_no_puffs", "before", "whole"]:
            for log in [True, False]:
                if base_key == "pre_alt" and not log:
                    continue
                mag = ie.s15_magnitudes(REST, base_key, bw, kw, log=log)
                v = ie.s15_verdict(mag)
                F15[(base_key, bw, kw, log)] = (mag, v)
                med = v["median"]
                rows.append(dict(baseline=("code session" if base_key == "pre" else "SD245 Preinfusion") + ", " + bw,
                                 ketamine=kw, power="log10" if log else "linear", largest=med.index[0],
                                 largest_value=med.iloc[0], ORB=med["ORB"], ORB_rank=v["rank"]["ORB"],
                                 IFG_rank=v["rank"]["IFG"], MFG_rank=v["rank"]["MFG"], median_region=med.median(),
                                 n_regions=v["n_regions"], ORB_first=v["orb_first"], frontal_above_median=v["frontal_above"],
                                 met=v["orb_first"] and v["frontal_above"]))
V15 = pd.DataFrame(rows)
print(V15.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
print("variants meeting the criterion:", int(V15.met.sum()), "of", len(V15), "; ORB first:", int(V15.ORB_first.sum()),
      "; ORB, IFG and MFG above the median region:", int(V15.frontal_above_median.sum()))
print("largest region across the variants:", dict(Counter(V15.largest).most_common()))

mag, v = F15[("pre", "before", "after", True)]
print("\nmain reading (baseline: code preinfusion session before the first puff; ketamine: first puff to end; log10):")
print(pd.DataFrame({"median": v["median"], "participants": v["n_pts"].reindex(v["median"].index),
                    "rank": pd.Series(v["rank"])}).T.round(3).to_string())
print("\nmagnitude by participant (rows) and region:\n", mag[v["median"].index].round(2).to_string())
prank = mag.rank(axis=1, ascending=False)
print("\nrank of ORB, IFG, MFG within each participant's regions (NaN: no channel), and the participant's number of regions:")
print(pd.DataFrame({r: prank[r] for r in ["ORB", "IFG", "MFG"]}).assign(n_regions=mag.notna().sum(1)).to_string())
fig, ax = plt.subplots(figsize=(8, 2.4))
ax.bar(range(len(v["median"])), v["median"].values, color=["C1" if r in ("ORB", "IFG", "MFG") else "C0" for r in v["median"].index])
ax.set_xticks(range(len(v["median"]))); ax.set_xticklabels(v["median"].index, rotation=90, fontsize=7); ax.set_ylabel("median |z|")
plt.tight_layout(); plt.show()
print("plotted range:", round(float(v["median"].min()), 3), "to", round(float(v["median"].max()), 3))

**Result: not reproduced, on approximate windows.** In the main reading (baseline the preinfusion file
before the first puff, ketamine the second half of the infusion file, about 30 to 40 min, log power), the
median magnitude is largest in MCC (2.644, 2 participants), then IFG (2.267, 6 participants), ORB (2.141, 5)
and MFG (2.048, 2), against a median region of 1.386 over 21 regions. The three frontal regions rank 2 to 4,
above the median region, but ORB is third, so the criterion fails on its first part. Over the 24 window,
baseline and power-scale variants, ORB, IFG and MFG are above the median region in all 24, and ORB is first
in 2 (both with the whole preinfusion file as baseline and the infusion file after the last puff, 16
windows); MCC is first in 20 and MFG in 2. Within participants, ORB is the largest region in SD172, SD181 and
SD186 and third in SD185 and SD245, the other two participants with ORB channels. MCC rests on SD185 (1
channel) and SD245 (14 channels). Cohort (b) cannot be separated from (a): the 7 participants with LFP files
are the included ones, and the dataset does not record the infusion type. The test is approximate: the
paper's 25-min baseline does not fit in the 20-min files, and the 30 to 40 min window is placed from the
Methods' "approximately 30 minutes" and the 10 min of recording before the first puff.

## figs27-slow-oscillations-with-strong-dissociation

> All participants with strong dissociation (CADSS above 10) show ketamine-induced oscillations below 10 Hz,
> and high-CADSS participants tend to have a larger fraction of channels with such oscillations.

The detection of the Methods (Band-limited oscillation detection; criterion), on the channels and
spectrograms of figs15: segment spectra are the mean of the 30-s multitaper windows inside each 3-min
segment; FOOOF removes the aperiodic part; `scipy.signal.find_peaks` finds the peaks; peaks are clustered
by 80% overlap of peak width; clusters in fewer than 40% of a window's segments are dropped; a ketamine
cluster below 10 Hz counts if no baseline cluster lies within 2 Hz (new frequency) or if a one-sided
rank-sum test of its peak heights against those of a baseline cluster within 2 Hz gives p < 0.001.
Details the paper does not give, each an assumption: FOOOF 'fixed' aperiodic mode with default settings,
fitted on 1 to 40 Hz; peaks above the aperiodic fit with a prominence of at least 0.1 log10 units (0.05
and 0.2 as variants); a peak's width at half its prominence; clusters by single linkage, two peaks
joined when their overlap is at least 80% of the narrower one's width ('min'; 80% of the wider one's,
'max', as a variant); one height per segment (the highest); a cluster's frequency the median of its peak
centres, which also decides "within 2 Hz" and "below 10 Hz".

Windows: baseline the preinfusion and postinfusion files combined (Methods: "pre- and post-infusion windows
were combined"; the whole 20-min files, 12 segments), with the preinfusion file alone (6 segments) and its
first half (3) as variants; ketamine window the second half of the infusion file (30 to 40 min, the
criterion's fallback, 3 segments), with the first half (3) and the whole file (6) as variants. No sleep or
ictal annotations exist, so none are excluded. As a reference without ketamine, the same detection tests
the second half of the postinfusion file against the preinfusion file.

CADSS: the infusion score per participant (maximum over the infusion time points, as the code's
`utils.load_full_cadss` and notebook 01) from the code record's `data/cadss_across_pts.csv`. High CADSS
under both readings of Open H, > 10 (Results) and >= 10 (Methods, fig. S27F). Cohorts as in figs15: only
(a) can be run. The label permutation of the Methods (not required) is enumerated over every assignment of
the high label.

In [ ]:
CAD = he.load_cadss().loc[cd.PARTICIPANTS, ["pre", "infusion", "post"]]
print("CADSS (code record), the 7 participants:\n", CAD.to_string())
HIGH = {">10": [p for p in cd.PARTICIPANTS if CAD.loc[p, "infusion"] > 10],
        ">=10": [p for p in cd.PARTICIPANTS if CAD.loc[p, "infusion"] >= 10]}
print("high CADSS:", HIGH)
t = time.time()
FL = ie.s27_flats(REST, {"pre": ["whole"], "post": ["whole", "after"], "inf": ["after", "before", "whole"]})
print(f"FOOOF fits: {sum(v[1].shape[0] for v in FL.values())} channel segments in {len(FL)} segment spectra, {time.time() - t:.0f} s; "
      f"non-finite values in the flattened spectra: {sum(int((~np.isfinite(v[1])).sum()) for v in FL.values())}")
import scipy.stats as sst
print("lowest one-sided rank-sum p reachable (no ties), ketamine against baseline segments:")
for nk, nb_ in [(3, 12), (3, 6), (3, 3), (6, 12), (6, 6), (6, 3)]:
    print(f"  {nk} against {nb_}: {sst.mannwhitneyu(np.arange(nb_, nb_ + nk), np.arange(nb_), alternative='greater').pvalue:.4g}")

In [ ]:
BASES = {"pre + post": [("pre", "whole"), ("post", "whole")], "pre": [("pre", "whole")], "pre first half": [("pre", "before")]}
def summarise(r, hi):
    n_osc = {p: r[p]["n_osc"] for p in r}; n_ch = {p: r[p]["n_ch"] for p in r}
    d, pp, nn = ie.label_permutation(n_osc, n_ch, hi)
    return sum(n_osc[p] > 0 for p in hi), d, pp, nn
t = time.time()
rows, S27 = [], {}
for bname, base in BASES.items():
    for kw in ["after", "before", "whole"]:
        for prom in [0.05, 0.1, 0.2]:
            for ov in ["min", "max"]:
                r = ie.s27_channels(REST, FL, base, kw, prom, ov)
                S27[(bname, kw, prom, ov)] = r
                p0 = cd.PARTICIPANTS[0]
                row = dict(baseline=bname, ketamine=kw, prominence=prom, overlap=ov,
                           segments=f"{r[p0]['n_base_seg']}/{r[p0]['n_ket_seg']}")
                for thr, hi in HIGH.items():
                    k, d, pp, nn = summarise(r, hi)
                    row[f"high{thr} with osc"] = f"{k}/{len(hi)}"
                    row[f"diff{thr}"] = d; row[f"perm_p{thr}"] = pp
                row.update({p: r[p]["n_osc"] for p in cd.PARTICIPANTS})
                rows.append(row)
V27 = pd.DataFrame(rows)
print(f"({time.time() - t:.0f} s) channels with a ketamine-unique oscillation below 10 Hz, by participant, for each variant;"
      " diff = fraction of channels in high minus low CADSS participants, perm_p its exact permutation p:")
print(V27.to_string(index=False, float_format=lambda x: f"{x:.3f}"))
for thr, hi in HIGH.items():
    ok = V27[f"high{thr} with osc"] == f"{len(hi)}/{len(hi)}"
    print(f"CADSS {thr}: variants in which every high-CADSS participant has at least one channel: {int(ok.sum())} of {len(V27)}")
print(f"(the label permutation has {summarise(S27[('pre + post', 'after', 0.1, 'min')], HIGH['>10'])[3]} assignments for >10 "
      f"and {summarise(S27[('pre + post', 'after', 0.1, 'min')], HIGH['>=10'])[3]} for >=10)")

In [ ]:
MAIN27 = S27[("pre + post", "after", 0.1, "min")]
rows = []
for p in cd.PARTICIPANTS:
    r = MAIN27[p]
    fr = [u[0] for h in r["hits"] for u in h[2]]
    kinds = Counter(u[1] for h in r["hits"] for u in h[2])
    rows.append(dict(participant=p, CADSS=CAD.loc[p, "infusion"], high_gt10=p in HIGH[">10"], high_ge10=p in HIGH[">=10"],
                     channels=r["n_ch"], with_osc=r["n_osc"], fraction=r["n_osc"] / r["n_ch"],
                     new=kinds.get("new", 0), amplified=kinds.get("amplified", 0),
                     freq_range=f"{min(fr):.1f} to {max(fr):.1f}" if fr else "",
                     regions=dict(Counter(h[1] for h in r["hits"]).most_common(4))))
T27 = pd.DataFrame(rows).set_index("participant")
print("main reading (baseline pre + post, ketamine 30 to 40 min, prominence 0.1, overlap of the narrower peak):")
print(T27.to_string(float_format=lambda x: f"{x:.3f}"))
for thr, hi in HIGH.items():
    k, d, pp, nn = summarise(MAIN27, hi)
    lo = [p for p in cd.PARTICIPANTS if p not in hi]
    fh = sum(MAIN27[p]["n_osc"] for p in hi) / sum(MAIN27[p]["n_ch"] for p in hi)
    fl_ = sum(MAIN27[p]["n_osc"] for p in lo) / sum(MAIN27[p]["n_ch"] for p in lo)
    print(f"CADSS {thr}: {k} of {len(hi)} high participants with >= 1 channel; channel fraction high {fh:.3f}, low {fl_:.3f}, "
          f"difference {d:.3f}, permutation p {pp:.3f} ({nn} assignments)")
print("\nall channels with an oscillation, by region (main reading):", dict(Counter(h[1] for p in cd.PARTICIPANTS for h in MAIN27[p]["hits"]).most_common()))

# reference without ketamine: postinfusion second half against the preinfusion file
REF = ie.s27_channels(REST, FL, [("pre", "whole")], "after", 0.1, "min", ket_key="post")
print("\nreference, postinfusion (second half) against preinfusion, channels with a 'unique' oscillation below 10 Hz:",
      {p: f"{REF[p]['n_osc']}/{REF[p]['n_ch']}" for p in cd.PARTICIPANTS})
same = S27[("pre", "after", 0.1, "min")]
print("the same baseline (preinfusion file) with the infusion second half:", {p: f"{same[p]['n_osc']}/{same[p]['n_ch']}" for p in cd.PARTICIPANTS})
above = {p: same[p]["n_osc"] > REF[p]["n_osc"] for p in cd.PARTICIPANTS}
for thr, hi in HIGH.items():
    print(f"CADSS {thr}: high participants with more channels on infusion than in the reference: {sum(above[p] for p in hi)} of {len(hi)}")
print("all participants:", sum(above.values()), "of", len(above), "; total channels, infusion", sum(same[p]["n_osc"] for p in cd.PARTICIPANTS),
      "against reference", sum(REF[p]["n_osc"] for p in cd.PARTICIPANTS))

**Result: reproduced as the criterion is written, on approximate windows; the detection is not specific
to ketamine.** In the main reading (baseline the preinfusion and postinfusion files, ketamine the second
half of the infusion file, prominence 0.1), every high-CADSS participant has at least one channel with a
ketamine-unique oscillation below 10 Hz: SD172 28 of 97 channels, SD186 4 of 47, SD211 6 of 44 and SD245 8
of 83 (CADSS > 10, 4 of 4), and SD181 33 of 58 (CADSS >= 10, 5 of 5). This holds in 50 of the 54 variants
under each threshold; the four exceptions all use the combined baseline with the wider-peak overlap rule,
and in each SD245 has none. All detected oscillations are at new frequencies: with 3 ketamine segments the
one-sided rank-sum test cannot reach p < 0.001 (lowest reachable p 0.0022 against 12 segments). The
low-CADSS participants also have such channels (SD168 5 of 86, SD185 2 of 43), and the same detection run on
the second half of the postinfusion file against the preinfusion file finds them in all 7 participants (60
channels, against 97 for the infusion file with the same baseline); only 2 of the 4 participants with CADSS
> 10 have more channels on infusion than in that reference. For the second part (not required), the channel
fraction is higher in high-CADSS participants under >= 10 (0.240 against 0.054, exact permutation p 0.048 over
21 assignments) and not under > 10 (0.170 against 0.214, p 0.629 over 35). The windows and the peak threshold
are assumptions, and sleep and ictal periods could not be excluded.

## Blocked claim

figs12-erp-not-explained-by-eyelid-closure needs the onsets of spontaneous blinks. The behaviour files hold
continuous eye-closure traces and puff events but no spontaneous blink onsets (printed in the data check);
the paper's code lists them by hand (`for_reference/analyze_airpuff_ieeg_natural_blinks.py`, lines 42 to
79, 52 onsets in the baseline sessions). The claims file says to block in that case; blocked, data_missing.

## Summary

| Claim | Outcome | Key numbers |
|---|---|---|
| fig2g-nmf-six-factors | reproduced (assumed elbow rule) | 458 channels; held-out EV 0.466 at six components; elbow at 6 by largest fall in gain, 8 by chord distance |
| link-ieeg-yeo-labels | reproduced | 458 of 458 channels found; 120 of 126 dataset network counts equal; deposited Fig. 2H and 2J matrices recomputed exactly |
| fig2h-factor-network-loading | partially reproduced | 4 of 6 rows; limbic (0.495) and frontoparietal (0.463) fall short |
| fig2i-ketamine-reduces-factors-1-2 | reproduced | factor 1 -0.125 (p 2.56e-05), factor 2 -0.115 (p 5.51e-08); factors 0, 4, 5 not significant; means within tolerance |
| fig2j-factor-change-by-network | reproduced (reading a) | all six named changes negative |
| figs13-auditory-control-no-response | reproduced | control below preinfusion for 6 of 6 factors (test-half preinfusion, ratios 0.33 to 0.70; all preinfusion trials, 0.34 to 0.98) |
| fig3b-cluster-test-yeo-networks | reproduced | limbic, ventral attention, somatomotor significant; maps equal the deposited ones |
| fig3c-cluster-test-regions | reproduced | 10 of 10 listed significant, 10 of 10 others not |
| figs15-ketamine-spectral-changes-frontal | not reproduced (approximate windows) | MCC largest (2.644), IFG 2.267, ORB third 2.141, MFG 2.048, median region 1.386; ORB first in 2 of 24 variants |
| figs12-erp-not-explained-by-eyelid-closure | blocked, data_missing | spontaneous blink onsets only in the code |
| figs27-slow-oscillations-with-strong-dissociation | reproduced as written (approximate windows) | 4 of 4 (CADSS > 10) and 5 of 5 (>= 10) high-CADSS participants with a channel; 50 of 54 variants; the postinfusion reference finds such channels in all 7 participants |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

Differences from the paper's method: every test starts from the dataset's precomputed spectrograms, not
from the LFP, so the multitaper, z-scoring, bipolar referencing and channel selection are taken as
deposited (the train and test means were shown to equal the means of the per-trial file). The Yeo7 label
of each channel comes from the code record. The elbow rule for fig2g is an assumption. For fig3c only
reading (a) was run. figs15 and figs27 are approximate tests on the 20-min LFP files: the 25-min
pre-ketamine baseline is replaced by the preinfusion eyepuff file, the peak ketamine window by the second
half of the infusion file (placed at about 30 to 40 min from the Methods' eyepuff timing), no sleep or
ictal periods are excluded, the fig. S15 spectrogram uses the fig. S27 multitaper parameters, and the FOOOF
range, peak threshold, peak width and clustering rule of figs27 are assumptions; CADSS scores come from the
code record.

What was found about the dataset: the aggregated products are complete and consistent with one another
and with the paper's deposited outputs, which this notebook regenerates exactly (NMF factors and loadings,
network and region t maps and contours). The per-channel Yeo7 labels are not in the dataset. The
per-participant LFP files are 20-min windows around each puff session (10 min before and after the first
puff), with no infusion start time and no sleep or seizure annotations; the behaviour files carry no
spontaneous blink onsets or CADSS scores. The network file's time bins sit
0.6 ms off the per-trial file's.

In [ ]:
print(f"notebook ran in {(time.time() - T0) / 60:.1f} min")